# A gate you already beat cannot measure anything

My offline gate was two opponents I beat 28/30 and 30/30. It was stable, it was paired, it was fast, and for three days it answered every question I asked it with "no difference."

That was not the answer. It was the gate having no games left to flip.

Four times in one day that gate returned the wrong sign. It called a real improvement flat, it hid a routing rule that was actively losing me games, it told me a whole family of tapes was useless when the best one in it was worth +42 wins per 200 games, and it closed a search direction that had five untouched holes in it. Here are all four with numbers, and the one-line check that would have caught it on day one.

## The gate

Two live opponents, 30 pinned towns each, one observation per town, paired.

| opponent | my agent |
|---|---|
| yhay81 Three-Day Shop Router | 28/30 |
| boatlee_v14 | 30/30 |

58 wins out of 60. Nothing wrong with the harness. The problem is arithmetic: there are two games in it that can move. Any change that is worth less than about 3% of win rate is invisible, and any change that is worth more than that can only show up as a regression.

## Failure 1 — it called a real improvement flat

I ported a sell-ordering rule from a public notebook: rank the sell orders in a turn by how much each one loses if the opponent's visible ripe yield is quoted ahead of it, rather than by headline revenue.

On the old gate: median **-10 coins**, one game flipped. I wrote it up as inert.

Then I added a mid-ladder opponent I was beating about 60% of the time:

| gate | before | after |
|---|---|---|
| yhay81 (28/30) | 28/30 | 29/30 |
| boatlee (30/30) | 30/30 | 30/30 |
| mid-ladder opponent | **19/30** | **22/30** |
| the same mid-ladder opponent, 60 unseen towns | **34/60** | **38/60** |

Same code. The old gate saw +1. The new one saw +7 over 180 games.

## Failure 2 — it hid a rule that was losing me games

My agent routes to a different recorded schedule depending on which shop the town opens first. One of those routes had been in every submission for a week.

Against the mid-ladder opponent, broken down by the town's first shop:

| town's first shop | my win rate |
|---|---|
| **ICE_CREAM_SHOP (a route I had added)** | **3/19 = 16%** |
| BAKERY (no route) | 22/23 = 96% |

The route was not helping. It was costing me four games in five, in a fifth of all towns. The old gate could not see it because in those same towns I still beat both of its opponents.

Replacing that one route, verified on 38 towns never used to pick it: **11/38 → 34/38**.

## Failure 3 — it closed a search that had five holes left

An earlier note in my own repo says, after a sweep of every shop group: *"only YARN is a hole."* That sweep ran on the saturated gate.

Re-run against an opponent near 50%, five groups had never been routed at all and four of them paid:

| group | before | after (all on held-out towns) |
|---|---|---|
| ICE_CREAM_SHOP | 11/38 | 34/38 |
| SMOOTHIE_SHOP | 15/26 | 23/26 |
| FARMERS_MARKET | 26/41 | 32/41 |
| BRUNCH_SPOT | 37/46 | 42/46 |
| YARN_STORE | 29/35 | 33/35 |

Total across 400 towns against that opponent: **283/400 → 370/400**.

## Failure 4 — the expensive one

Three days earlier I had tested 249 recorded schedules from eight top-20 teams as my base schedule and written down: *"all lost by 53 to 107 games."* On the saturated gate, that is what they do.

On a gate built from opponents I actually meet near my own rating, the base schedule turned out to be the single largest lever I have, larger than every routing change combined:

| measurement | old base | a top-2 team's base |
|---|---|---|
| band gate, 30 towns never used to pick it | 52/90 | **86/90** |
| ten band opponents, 200 games | 137/200 | **179/200** |
| the strongest public agent in the competition | 0/30 | **4/30** |
| one opponent I had never beaten | 11/30 | **29/30** |

The hardest opponents flipped outright: 8/20 to 20/20, 8/20 to 17/20, 8/20 to 18/20.

Two guards worth keeping from that one. The winning base scores **lower** than mine in solo play (133,217 against 145,335) so picking by solo money would have discarded it, and it is only strong in combination: run it with my routing removed and it drops to **24/200**.

## The check

Before trusting a gate, ask what fraction of it you already win. The number that matters is not the win rate, it is how many games are still available to move in each direction.

A gate at 97% can report a regression and nothing else. A gate at 50% resolves both directions and resolves them fastest, because that is where the variance of a win count is largest.

This is not a statement about my competition. Any paired offline harness has it: held-out AUC on a split you already score 0.99 on, a regression suite that passes, a benchmark where you lead. The failure mode is quiet, because a saturated gate does not return an error. It returns "no difference," which reads exactly like a negative result.

In [ ]:
def gate_headroom(wins, n, effect_pp=5.0):
    """How much room is left in a gate, and the smallest effect it can resolve.

    wins, n   : your current record on the gate
    effect_pp : the improvement you care about, in percentage points of win rate

    `room_up` is the number of games that could still flip your way. If the effect
    you are hunting is bigger than that, the gate literally cannot show it to you.
    """
    rate = wins / n
    room_up, room_down = n - wins, wins
    detectable = 100.0 * room_up / n
    # a rough two-sided resolution floor: one standard error of the win count
    se_pp = 100.0 * (rate * (1 - rate) / n) ** 0.5
    verdict = ("SATURATED - can only report regressions" if effect_pp > detectable else
               "coarse - the effect is near the gate's own noise" if effect_pp < se_pp else
               "usable")
    return {
        "win rate": f"{100 * rate:.0f}%",
        "games that can still flip up": room_up,
        "largest gain the gate can show": f"{detectable:.0f} pp",
        "one standard error": f"{se_pp:.1f} pp",
        "verdict": verdict,
    }


# my gate for three days, and the one that replaced it
for label, (w, n) in {"old gate": (58, 60), "added a 50% opponent": (77, 90)}.items():
    print(f"{label:24}", gate_headroom(w, n, effect_pp=5.0))

## What I do now

I keep the saturated opponents, because they are still the cheapest regression detector I have: if a change breaks something badly, they catch it. But nothing gets **selected** on them any more.

For selection I pull the actual opponents I met on the ladder, take the ones near my own rating, and gate on those. In my case that gate reproduced my real ladder win rate to within two points: 49% offline against 51% measured over 173 real games.

The habit that produced all four failures was reading "no difference" as information. It is only information if the gate could have said otherwise.

If you have a gate you win 90%+ of and a result from it you are relying on, that is the one I would re-run first. I would be glad to hear if it holds up; mine did not, four times.